# Genre Bias in the Academy Awards: A Statistical Analysis of Oscar Winning Films

**Group Members:** Edison Ayran

In [14]:
# libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.stats import chi2_contingency
from statsmodels.sandbox.stats.runs import runstest_1samp
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

# Introduction

This project investigates whether the genre of a film influences its probability of winning an Academy Award. The Academy Awards are widely regarded as one of the most prestigious honors in the film industry, and winning an Oscar can significantly impact a film’s cultural recognition, financial success, and the careers of those involved. Because of this influence, it is important to examine whether certain types of films are systematically favored in award outcomes. Using data from the Academy Awards database and the IMDb dataset, this analysis explores whether genres such as Drama or Biography which are often considered “prestige genres,” are more likely to win awards than genres such as Action, Comedy, or Horror. By combining award data with film metadata, this project aims to statistically evaluate whether genre plays a meaningful role in Oscar success.

# Problem Statement

Despite the perception that Academy Awards celebrate the “best” films of the year, critics have long argued that certain genres are consistently favored by voters. In particular, dramatic or biographical films appear to receive disproportionate recognition compared to genres such as comedy, action, or horror. The goal of this project is to investigate whether this perceived genre bias exists using statistical analysis. By examining nomination and winning outcomes across multiple award categories and comparing them with film genre classifications, we aim to determine whether genre significantly affects a film’s probability of winning an Academy Award. Understanding these patterns can provide insight into whether award outcomes reflect purely artistic merit or whether institutional preferences for specific storytelling styles influence the results.

# Objective

Determine whether movie genre significantly influences the probability of winning an Academy
Award across four major film-level categories (Best Picture, Directing, Original Screenplay,
Adapted Screenplay) from 2004–2024, and test whether this relationship has changed
meaningfully between the pre-reform era (2004–2014) and the post-reform era (2015–2024)
using chi-squared tests of independence, the Wald-Wolfowitz runs test, and logistic
regression with genre indicator variables and genre × year interaction terms.

### Research Question

**Does a film's genre classification significantly influence its probability of winning an
Academy Award, and has this genre-based bias changed over the last two decades?**

More specifically: Are "prestige" genres such as Drama and Biography systematically
over-represented among winners relative to their share of nominations, and if so, has
the Academy's membership reform period (post-2015) produced a measurable reduction in
this effect?

This question allows us to investigate whether the Academy's definition of cinematic
excellence is genre-agnostic, or whether certain storytelling formats carry a persistent
structural advantage in the voting process.

# Description of Variables

**From `full_data.csv`:**

| Column              | Type    | Description |
|---------------------|---------|-------------|
| `FilmId`            | string  | IMDb tconst (e.g. `tt0081398`) - primary merge key with IMDb datasets |
| `Film`              | string  | Title of the nominated film |
| `Year`              | string  | Ceremony year (e.g. `2004/05`) |
| `CanonicalCategory` | string  | Standardized category name - filtered to our 4 target categories |
| `Category`          | string  | Raw category label as listed in the official Academy database |
| `Winner`            | boolean | True if the film won; NaN treated as False (nominated but did not win) |
| `Name`              | string  | Person associated with the nomination (director or writer) |

**From `title.basics.tsv.gz`:**

| Column           | Type   | Description |
|------------------|--------|-------------|
| `tconst`         | string | IMDb title identifier - merge key |
| `genres`         | string | Comma-separated genre tags, up to 3 per film (e.g. `Drama,Biography,History`) |
| `runtimeMinutes` | float  | Film runtime in minutes - regression control variable |
| `titleType`      | string | Used to pre-filter to movies only before merging |
| `startYear`      | int    | Release year - used to cross-validate ceremony year |

**From `title.ratings.tsv.gz`:**

| Column          | Type   | Description |
|-----------------|--------|-------------|
| `tconst`        | string | IMDb title identifier - merge key |
| `averageRating` | float  | Weighted average IMDb user rating (scale 1–10) |
| `numVotes`      | int    | Total number of user ratings received |

**Engineered during cleaning:**

| Column          | Type   | Description |
|-----------------|--------|-------------|
| `primary_genre` | string | First genre extracted from `genres` - core categorical predictor in all 3 tests |
| `outcome`       | int    | Binary re-encoding of `Winner` - 1 if won, 0 if nominated only |
| `era`           | int    | 0 = pre-reform (2004–2014), 1 = post-reform (2015–2024) |
| `log_votes`     | float  | Log-transformed `numVotes` to reduce right skew before regression |

## Dataset Composition

Our analysis uses three files merged via IMDb tconst identifiers.

**`full_data.csv` (Primary Oscar Source)**
Contains 12,014 nomination records across all Academy Award categories from 1927–2025.
Each record includes a `FilmId` column with an IMDb tconst (e.g. `tt0081398`),
enabling a clean exact join with IMDb
After filtering to our 4 target categories and ceremony years 2004–2024:

| Category                       | Nominations | Winners |
|--------------------------------|-------------|---------|
| BEST PICTURE                   | 171         | 21      |
| DIRECTING                      | 105         | 21      |
| WRITING (Original Screenplay)  | 105         | 21      |
| WRITING (Adapted Screenplay)   | 105         | 21      |
| **Total**                      | **486**     | **84**  |

**`title.basics.tsv.gz` (IMDb)**
Covers ~10 million IMDb titles. Provides genre classifications (up to 3 per film),
runtime in minutes, title type, and release year. Filtered to `titleType == 'movie'`
before merging.

**`title.ratings.tsv.gz` (IMDb)**
Provides weighted average user rating and total vote count for all rated titles.
Joined to basics on `tconst`, then merged with the Oscar records on `FilmId = tconst`.

Final working dataset after all merges: **486 nomination records**, 10 analytical columns.

## Load the Data ##

In [15]:
# Full Oscar dataset 
oscars_raw = pd.read_csv('full_data.csv', sep='\t', low_memory=False)

# IMDb title.basics 
imdb_basics = pd.read_csv(
    'title.basics.tsv.gz',
    sep='\t',
    compression='gzip',
    low_memory=False,
    na_values='\\N'
)

# IMDb title.ratings 
imdb_ratings = pd.read_csv(
    'title.ratings.tsv.gz',
    sep='\t',
    compression='gzip',
    na_values='\\N'
)

print("Oscar records :", oscars_raw.shape)   
print("IMDb basics   :", imdb_basics.shape)
print("IMDb ratings  :", imdb_ratings.shape)

oscars_raw.head(3)
imdb_basics.head(3)
imdb_ratings.head(3)

Oscar records : (12014, 16)
IMDb basics   : (12334462, 9)
IMDb ratings  : (1642919, 3)


,tconst,averageRating,numVotes
0,tt0000001,5.7,2197
1,tt0000002,5.5,309
2,tt0000003,6.5,2302
